<a href="https://colab.research.google.com/github/salinela/carbon-portfolio-project-v2/blob/main/notebooks/08_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# set up: desktop
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import sqlite3
import time
import seaborn as sns

# path set up:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("root on path:", ROOT)

from src.config import DATA_RAW, DATA_PROCESSED
from src import eda

# database connection set up:
DB = ROOT/'data/carbon.db'
con = sqlite3.connect(DB)
con.execute("PRAGMA foreign_keys = ON;")

In [1]:
# set up: Google Colab
import sys
import sqlite3
import time
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns

In [2]:

# mount drive (data artifacts live here — never in git)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# clone fresh, or pull if it already exists (so re-running the cell doesn't error)
import os
REPO = "/content/repo"
if os.path.exists(REPO):
    !cd {REPO} && git pull
else:
    !git clone https://github.com/salinela/carbon-portfolio-project-v2.git {REPO}
%cd {REPO}

Cloning into '/content/repo'...
remote: Enumerating objects: 161, done.
remote: Counting objects: 100% (161/161), done.
remote: Compressing objects: 100% (114/114), done.
remote: Total 161 (delta 90), reused 99 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (161/161), 311.84 KiB | 19.49 MiB/s, done.
Resolving deltas: 100% (90/90), done.
/content/repo


In [4]:
# root on path — mirrors desktop's package-style imports
sys.path.insert(0, REPO)
print("root on path:", REPO)

root on path: /content/repo


In [5]:
# live-reload src edits after a git pull without restarting the runtime
from src import eda
from src import feature_engineering as fe
from src.config import DATA_RAW, DATA_PROCESSED

In [6]:
# DB copied to LOCAL disk (not the Drive FUSE mount) to avoid SQLite locking.
# Needed to read/query it, not just to rebuild — copy once per session.
DRIVE = "/content/drive/MyDrive/carbon_project_v2"
if not os.path.exists("/content/carbon.db"):
    !cp "{DRIVE}/carbon.db" /content/carbon.db

In [7]:
DB = "/content/carbon.db"
con = sqlite3.connect(DB)
con.execute("PRAGMA foreign_keys = ON;")
print("fe MIN_PERIODS_FRAC:", fe.MIN_PERIODS_FRAC, "| batched:", hasattr(fe, "build_price_features_batched"))

fe MIN_PERIODS_FRAC: 0.7 | batched: True


one-time usage

In [ ]:
# %% auto-reload edited src modules (so src/*.py edits take effect without kernel restart)
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# one-time cleanup of the stale broken view
con.execute("DROP VIEW IF EXISTS v_company_emissions;")
con.commit()

# Phase 0: Data Assembly

In [8]:
meta, meta_diag = eda.build_meta(con)
fy,   fy_diag   = eda.build_firm_year(con)

meta_diag, fy_diag

({'n_companies': 8288,
  'by_universe': {'EU': 7988, 'ETS': 300},
  'eligible_n': 5883,
  'sector_nulls_master': 58,
  'sector_nulls_after_backfill': 58,
  'country_nulls': 0,
  'bvd_nulls': 0,
  'coverage_status_counts': {'mapped_loaded': 5883,
   'unmapped_exchange': 1773,
   'mapped_no_data': 523,
   'no_ticker': 109}},
 {'n_rows': 12487,
  'n_companies': 1306,
  'year_range': (2012, 2025),
  'by_source': {'trucost': 9214, 'ets_registry': 3273},
  'revenue_nulls': 573,
  'intensity_nulls': 573})

In [9]:
fy_ids = set(fy["company_id"])
elig   = meta[meta["eligible"] == 1]
mask   = elig.index.isin(fy_ids)
print("eligible:", len(elig))
print("eligible w/ emissions:", int(mask.sum()))
print(elig[mask]["universe"].value_counts().to_dict())

eligible: 5883
eligible w/ emissions: 1259
{'EU': 978, 'ETS': 281}


## Phase 1: Universe Characterization

### Section A: Compute carbon itensity tiers (source registry x 1-digit NACE x year)

In [10]:
# --- cohort flags on meta (enables optional carbon-blind comparison) ---
fy_ids = set(fy["company_id"])
meta["has_emissions_data"] = meta.index.isin(fy_ids).astype(int)
meta["carbon_sample"] = ((meta["eligible"] == 1) &
                         (meta["has_emissions_data"] == 1)).astype(int)

# cross-check against master's has_emissions flag
print("has_emissions agree:",
      (meta["has_emissions"] == meta["has_emissions_data"]).mean())
print("carbon_sample n:", int(meta["carbon_sample"].sum()))

has_emissions agree: 0.9639237451737451
carbon_sample n: 1259


In [11]:
# --- nace1 for tiering: master, backfilled from orbis, first digit ---
nace_full = meta["nace_code"].fillna(meta["orbis_nace_code"])
nace1 = nace_full.astype("string").str.extract(r"(\d)")[0]   # index = company_id

# --- recompute tiers ---
fy, tier_diag = eda.compute_tiers(fy, nace1)
tier_diag

{'tier_counts': {'non_ets_high': 2947,
  'non_ets_low': 2911,
  'non_ets_medium': 2872,
  'ets_high': 1074,
  'ets_low': 1044,
  'ets_medium': 1015,
  <NA>: 573,
  'ets_untiered': 41,
  'non_ets_untiered': 10},
 'nace1_nulls': 0,
 'n_untiered': 51,
 'n_tiered': 11863}

In [12]:
d = meta[meta["has_emissions"] != meta["has_emissions_data"]]
print(len(d))
print(d.groupby(["has_emissions", "has_emissions_data"]).size())
print(d["universe"].value_counts().to_dict())

299
has_emissions  has_emissions_data
0              1                     299
dtype: int64
{'ETS': 299}


### Section B: Monthly Tier-based Portfolio Returns Helper

Main objective: for each month, take every firm currently sitting per tier and average their forward returns (equal-weighted basket)

tier_portfolio_returns produces one such series per tie; "do high-carbon baskets earn different returns than low-carbon ones"

The attach_tier_asof step is what tells each company-month which basket it was in at that date, using the 1-July lag so you're never using an emissions figure before it was public.

In [13]:
# panel load — desktop: your processed dir; Colab: DRIVE

features = pd.read_parquet("/content/drive/MyDrive/carbon_project_v2/features_month_end.parquet")
label    = pd.read_parquet("/content/drive/MyDrive/carbon_project_v2/label_fwd_return.parquet")

In [14]:
# pivot to wide:
panel = features.pivot_table(index=["company_id", "date"],
                             columns="signal_name", values="value")

panel = panel.join(label.set_index(["company_id", "date"])["fwd_ret"])
print("panel:", panel.shape)

panel: (730161, 42)


In [15]:

panel_t = eda.attach_tier_asof(panel, fy)
print("tier coverage:", round(panel_t["carbon_tier"].notna().mean(), 3))

tier coverage: 0.184


In [16]:
tret = eda.tier_portfolio_returns(panel_t)
print(tret.shape)
tret.tail()

(155, 8)


carbon_tier,ets_high,ets_low,ets_medium,ets_untiered,non_ets_high,non_ets_low,non_ets_medium,non_ets_untiered
date,,,,,,,,
2026-01-30,NaN,NaN,NaN,NaN,0.001464,0.020621,-0.019596,NaN
2026-02-27,NaN,NaN,NaN,NaN,-0.057135,-0.056221,-0.082630,NaN
2026-03-31,NaN,NaN,NaN,NaN,0.044266,0.047350,0.072799,NaN
2026-04-30,NaN,NaN,NaN,NaN,0.043705,0.013508,0.048464,NaN
2026-05-29,NaN,NaN,NaN,NaN,-0.027886,-0.015544,-0.011606,NaN


In [17]:
# 1) firms actually CONTRIBUTING per tier per year (drives tret NaN directly)
chk = (panel_t.reset_index()[["company_id", "date", "carbon_tier", "fwd_ret"]]
       .dropna(subset=["carbon_tier", "fwd_ret"]))

# remove untiered:
chk = chk[~chk["carbon_tier"].astype(str).str.endswith("untiered")]

# see number of companies in each yearly tier:
chk["year"] = pd.to_datetime(chk["date"]).dt.year
chk.groupby(["year", "carbon_tier"])["company_id"].nunique().unstack("carbon_tier")

carbon_tier,ets_high,ets_low,ets_medium,non_ets_high,non_ets_low,non_ets_medium
year,,,,,,
2013,68.0,75.0,70.0,NaN,NaN,NaN
2014,81.0,87.0,92.0,144.0,148.0,140.0
2015,83.0,81.0,81.0,182.0,176.0,182.0
2016,87.0,86.0,83.0,183.0,177.0,189.0
2017,86.0,86.0,79.0,273.0,268.0,285.0
2018,91.0,86.0,83.0,304.0,307.0,316.0
2019,94.0,88.0,89.0,312.0,312.0,326.0
2020,95.0,91.0,93.0,327.0,334.0,353.0
2021,94.0,85.0,87.0,347.0,364.0,380.0


In [18]:
# 2) is the bottleneck upstream (emissions/revenue) or the price panel?
fyt = fy[fy["carbon_tier"].notna()
         & ~fy["carbon_tier"].astype(str).str.endswith("untiered")]

# number of companies per year in the yearly dataframe:
fyt.groupby(["year", "source"]).size().unstack("source")

source,ets_registry,trucost
year,,
2012,246.0,NaN
2013,260.0,446.0
2014,266.0,472.0
2015,267.0,489.0
2016,272.0,769.0
2017,274.0,825.0
2018,275.0,846.0
2019,273.0,913.0
2020,272.0,936.0


In [19]:
tret_trim = tret.loc["2014-07-30":"2025-06-30"].drop(columns=['ets_untiered', 'non_ets_untiered'])
tret_trim.isna().mean()

,0
carbon_tier,
ets_high,0.015152
ets_low,0.030303
ets_medium,0.030303
non_ets_high,0.000000
non_ets_low,0.015152
non_ets_medium,0.015152


In [20]:
tret_trim[tret_trim.isna().any(axis = 1)]

carbon_tier,ets_high,ets_low,ets_medium,non_ets_high,non_ets_low,non_ets_medium
date,,,,,,
2018-02-28,-0.035744,NaN,NaN,-0.031747,-0.014578,0.003955
2018-03-30,0.000761,NaN,NaN,-0.028747,0.063794,0.038382
2024-02-29,NaN,NaN,NaN,0.084667,NaN,NaN
2024-03-29,NaN,NaN,NaN,-0.023028,NaN,NaN


In [24]:
sub

,company_id,date,amihud,beta_252,beta_63,boll_bw_20,boll_bw_60,boll_pctb_20,boll_pctb_60,brent_beta_252,...,stoch_k_14,stoch_k_63,trend_dist_sma200,trend_sma50_200,us10y_beta_252,us10y_beta_63,williams_r_14,williams_r_63,fwd_ret,carbon_tier
286,AT000000STR1,2018-02-28,0.029357,0.582807,0.863065,0.100220,0.129832,0.310648,0.195456,0.099389,...,29.545514,42.241398,-0.081875,-0.037619,0.035302,0.106512,-70.454486,-57.758602,NaN,ets_low
287,AT000000STR1,2018-03-30,0.029606,0.553461,0.514947,0.066408,0.147754,NaN,NaN,0.075077,...,NaN,NaN,NaN,-0.050252,0.027321,0.061778,NaN,NaN,NaN,ets_low
358,AT000000STR1,2024-02-29,0.028684,0.489384,0.925150,0.067240,0.186688,0.510240,0.686164,0.008327,...,47.999878,82.894702,0.111085,0.093975,-0.018506,-0.014942,-52.000122,-17.105298,NaN,ets_low
359,AT000000STR1,2024-03-29,0.031714,0.564581,1.090353,0.178147,0.137939,NaN,NaN,0.010626,...,NaN,NaN,NaN,0.080675,-0.012977,0.070633,NaN,NaN,NaN,ets_low
593,AT00000FACC2,2018-02-28,0.010240,1.550580,2.045107,0.173840,0.456735,1.071047,0.853758,-0.097629,...,97.461920,94.239632,0.617050,0.449148,0.150733,0.122241,-2.538080,-5.760368,NaN,non_ets_medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
729719,US8964385046,2018-02-28,0.239783,0.134265,-0.376984,0.063234,0.180888,0.326615,0.586343,0.224329,...,28.205078,64.925361,0.004362,-0.009358,-0.001652,0.033348,-71.794922,-35.074639,NaN,non_ets_medium
729790,US8964385046,2024-02-29,1.508199,0.617448,-1.387265,1.950531,1.090905,0.094769,-0.239007,0.421085,...,0.597016,0.546076,-0.865616,-0.322391,0.011222,0.395130,-99.402984,-99.453924,NaN,non_ets_high
729832,US89686D3035,2018-02-28,0.004351,0.014393,0.603566,0.096548,0.311571,0.281238,0.611673,-0.142275,...,29.000015,38.333342,-0.377166,-0.385668,-0.035920,0.020395,-70.999985,-61.666658,NaN,non_ets_medium
729903,US89686D3035,2024-02-29,0.094538,-0.734270,0.443706,0.088132,0.103921,0.450452,0.612902,-0.522167,...,51.351350,48.148150,-0.869369,-0.870606,0.137300,0.136023,-48.648650,-51.851850,NaN,non_ets_high


In [22]:
probe = ["2018-02-28", "2018-03-30", "2024-02-29", "2024-03-29"]
px = panel_t.reset_index()
px["date"] = pd.to_datetime(px["date"]).astype(str)

# companies in specific dates
sub = px[px["date"].isin(probe) & px["carbon_tier"].notna()
         & ~px["carbon_tier"].astype(str).str.endswith("untiered")]

In [23]:

# firms WITH a tier vs firms that also have a non-NaN fwd_ret, per tier-month
have_tier = sub.groupby(["date", "carbon_tier"])["company_id"].nunique()

have_ret  = (sub.dropna(subset=["fwd_ret"])
             .groupby(["date", "carbon_tier"])["company_id"].nunique())
pd.concat({"has_tier": have_tier, "has_ret": have_ret}, axis=1).fillna(0)

has_tier  has_ret
date       carbon_tier                      
2018-02-28 ets_high              84     14.0
           ets_low               85      1.0
           ets_medium            73      4.0
           non_ets_high         247     13.0
           non_ets_low          257      6.0
           non_ets_medium       243     17.0
2018-03-30 ets_high              63     14.0
           ets_low               45      1.0
           ets_medium            40      4.0
           non_ets_high         131     13.0
           non_ets_low          136      6.0
           non_ets_medium       149     17.0
2024-02-29 ets_high              77      2.0
           ets_low               76      0.0
           ets_medium            76      0.0
           non_ets_high         302      5.0
           non_ets_low          307      3.0
           non_ets_medium       305      1.0
2024-03-29 ets_high              34      2.0
           ets_low               25      0.0
           ets_medium            26      0.0
           non_ets_high          90      5.0
           non_ets_low           78      3.0
           non_ets_medium        95      1.0

A 1-month-forward return at end-Feb needs a price at end-Mar; end-March 2018 and end-March 2024 are Good Friday (2018-03-30, 2024-03-29 — markets shut). So:

At end-Feb, fwd_ret looks one month ahead to a closed Good Friday → NaN.
At end-Mar (the Good Friday itself), there's no price today → NaN.

That's why it's always a Feb/Mar pair, and only in years where Good Friday lands on the month-end. It's a forward-looking gap in the target, not a hole in the features. Which is why widening the window never helped — the NaN is baked into how the label was constructed.